In [10]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt

In [11]:
df = pd.read_csv("../data/processed/poi_raw_malang.csv")
print(f"Loaded Data: {len(df)} row")
print(f"Categories: {df['category'].value_counts().to_dict()}")

Loaded Data: 1554 row
Categories: {'place_of_worship': 633, 'school': 447, 'residential': 327, 'park': 58, 'marketplace': 31, 'college': 30, 'university': 28}


In [12]:
CATEGORY_MAP = {
    "n_campus":            ["university", "college"],
    "n_school":            ["school"],
    "n_market":            ["marketplace"],
    "n_place_of_worship":  ["place_of_worship"],
    "n_park":              ["park", "playground"],
    "n_residential":       ["residential"],
}

In [13]:
# Bounding box Malang City
LAT_MIN, LAT_MAX = -8.020, -7.890
LON_MIN, LON_MAX =  112.570, 112.700

STEP = 0.0045

# Generate grid points
lats = np.arange(LAT_MIN, LAT_MAX, STEP)
lons = np.arange(LON_MIN, LON_MAX, STEP)

grid_rows = []
grid_counter = 1

for lat in lats:
    for lon in lons:
        grid_rows.append({ 
            "grid_id": f"G{grid_counter:04d}",
            "lat": lat,
            "lon": lon,
        })
        grid_counter += 1

grid_df = pd.DataFrame(grid_rows)
print(f"Generated grid with {len(grid_df)} cells (500m x 500m)")


Generated grid with 841 cells (500m x 500m)


In [14]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # earth radius in meters
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * asin(sqrt(a))

In [15]:
RADIUS_M = 500 # 500 meter radius

for col in CATEGORY_MAP:
    grid_df[col] = 0

poi_by_group = {}
for col, categories in CATEGORY_MAP.items():
    poi_by_group[col] = df[df["category"].isin(categories)].copy()

for idx, row in grid_df.iterrows():
    if idx % 100 == 0:
        print(f"Processing grid {idx+1}/{len(grid_df)}")
    
    for col, poi_subset in poi_by_group.items():
        if poi_subset.empty:
            continue

        distances = poi_subset.apply(
            lambda p: haversine(row["lat"], row["lon"], p["lat"], p["lon"]),
            axis=1
        )
        grid_df.at[idx, col] = int((distances <= RADIUS_M).sum())

print("\fFinished calculating all grids")

Processing grid 1/841


Processing grid 101/841
Processing grid 201/841
Processing grid 301/841
Processing grid 401/841
Processing grid 501/841
Processing grid 601/841
Processing grid 701/841
Processing grid 801/841
Finished calculating all grids


In [16]:
# dominant zone classification
def classify_zone(row):
    scores = {
        "campus_area"      : row["n_campus"] * 5,
        "school_area"      : row["n_school"] * 2,
        "residential_area" : row["n_residential"] * 3,
        "commercial_area"  : row["n_market"] * 4,
        "worship_area"     : row["n_place_of_worship"] * 1,
        "public_area"      : row["n_park"] * 3,
    }

    max_score = max(scores.values())
    if max_score == 0:
        return "not classified"
    return max(scores, key=scores.get)

grid_df["dominant_zone"] = grid_df.apply(classify_zone, axis=1)

In [17]:
count_cols = list(CATEGORY_MAP.keys())
grid_filtered = grid_df[grid_df[count_cols].sum(axis=1) > 0].copy()
 
print(f"\nGrids with at least 1 POI: {len(grid_filtered)} out of {len(grid_df)} total")
print(f"\nDominant zone distribution:")
print(grid_filtered["dominant_zone"].value_counts())


Grids with at least 1 POI: 436 out of 841 total

Dominant zone distribution:
dominant_zone
residential_area    195
school_area         124
worship_area         70
campus_area          37
public_area          10
Name: count, dtype: int64


In [18]:
grid_df.to_csv("../data/processed/poi_grid_malang.csv", index=False)
 
grid_filtered.to_csv("../data/processed/poi_grid_malang_filtered.csv", index=False)
 
print(f"\n✅ Saved:")
print(f"   data/processed/poi_grid_malang.csv          ({len(grid_df)} rows)")
print(f"   data/processed/poi_grid_malang_filtered.csv ({len(grid_filtered)} rows)")
print(f"\nPreview first 5 rows (filtered):")
print(grid_filtered.head().to_string())


✅ Saved:
   data/processed/poi_grid_malang.csv          (841 rows)
   data/processed/poi_grid_malang_filtered.csv (436 rows)

Preview first 5 rows (filtered):
   grid_id   lat       lon  n_campus  n_school  n_market  n_place_of_worship  n_park  n_residential     dominant_zone
10   G0011 -8.02  112.6150         0         1         0                   0       0              1  residential_area
11   G0012 -8.02  112.6195         0         2         0                   3       0              3  residential_area
12   G0013 -8.02  112.6240         0         1         0                   3       0              2  residential_area
13   G0014 -8.02  112.6285         0         0         0                   1       0              3  residential_area
14   G0015 -8.02  112.6330         0         5         0                   1       0              1       school_area
